# Custom YOLOv11n Training

Trains a YOLOv11n model to detect faces for DJI Tello face-tracking.
Runs on Google Colab with a GPU.

## 1. Install Ultralytics and Mount Google Drive

In [1]:
%%capture

!pip install -U ultralytics

from google.colab import drive
drive.mount('/content/drive')

## 2. Prepare Your Dataset

The dataset is not included for privacy reasons, so use your own in YOLO format:

```
Face-Dataset/
├── images/
├── labels/
└── data.yaml
```

Zip it, upload to Google Drive, then change the path below to match yours.

In [2]:
!unzip -q "/content/drive/My Drive/Colab_Notebooks/Face-Dataset.zip" -d /content/Face-Dataset

## 3. Split into Train, Validation and Test Sets

Randomly splits the images about 70/15/15 so the model is tested on images it never trained on.

In [3]:
import random, yaml
from ultralytics.data.split import autosplit

random.seed(42)
autosplit("/content/Face-Dataset/images", weights=(0.7, 0.15, 0.15))

cfg = yaml.safe_load(open("/content/Face-Dataset/data.yaml"))
cfg.update({"path": "/content/Face-Dataset", "train": "autosplit_train.txt",
            "val": "autosplit_val.txt", "test": "autosplit_test.txt"})
yaml.safe_dump(cfg, open("/content/Face-Dataset/data.yaml", "w"))

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Autosplitting images from /content/Face-Dataset/images
: 100% ━━━━━━━━━━━━ 142/142 4.3Kit/s 0.0s


## 3. Load Pretrained YOLOv11n

In [4]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

## 4. Train the Model

Adjust epochs, image size and batch size to fit your dataset and GPU.

In [5]:
results = model.train(
    data="/content/Face-Dataset/data.yaml",
    epochs=20,
    imgsz=320,
    batch=16,
    device=0
)

Ultralytics 8.4.197 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Face-Dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=None, opset=None, optimize=False, opti

## 6. Evaluate on the Test Set

Tests the best model on images it never saw during training.

In [6]:
best = YOLO(f"{model.trainer.save_dir}/weights/best.pt")
metrics = best.val(data="/content/Face-Dataset/data.yaml", split="test")
print(f"Test mAP50: {metrics.box.map50:.3f} | mAP50-95: {metrics.box.map:.3f}")

Ultralytics 8.4.197 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,597,051 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1446.0±343.9 MB/s, size: 93.9 KB)
val: Scanning /content/Face-Dataset/labels... 25 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 25/25 1.5Kit/s 0.0s
val: New cache created: /content/Face-Dataset/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 1.3it/s 1.6s
                   all         25         25          1      0.998      0.995      0.779
Speed: 2.2ms preprocess, 30.0ms inference, 0.0ms loss, 3.4ms postprocess per image
Results saved to /content/runs/detect/val
Test mAP50: 0.995 | mAP50-95: 0.779


## 7. Save the Best Weights

Copies the best model so you can use it for Tello face-tracking.

In [7]:
!cp runs/detect/train/weights/best.pt /content/custom-yolo11n.pt